# 01. 탐색적 데이터 분석 (EDA)
**Credit Risk Dataset — 대출 금리 예측 프로젝트**

---
### 분석 목표
1. 데이터 기본 구조 파악 (shape, dtype, 결측치)
2. 목표변수(`loan_int_rate`) 분포 확인
3. 범주형 / 수치형 변수 분포 탐색
4. 피처 간 상관관계 분석 (다중공선성)
5. 목표변수와 각 피처 간의 관계 시각화
6. 주요 인사이트 정리

---
## 0. 환경 설정

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

ROOT     = os.path.abspath('..')
RAW_PATH = os.path.join(ROOT, 'credit_risk_dataset.csv')

print('설정 완료')

---
## 1. 데이터 로드 및 기본 정보

In [ ]:
df = pd.read_csv(RAW_PATH)
print(f'Shape: {df.shape}  →  {df.shape[0]:,}행 x {df.shape[1]}컬럼')
df.head()

In [ ]:
df.tail()

In [ ]:
# 컬럼별 dtype 및 non-null 개수
df.info()

In [ ]:
# 수치형 컬럼 기술통계
df.describe().T.style.background_gradient(cmap='Blues', subset=['mean','std','50%'])

---
## 2. 결측치 분석

In [ ]:
miss     = df.isnull().sum().rename('결측치 수')
miss_pct = (miss / len(df) * 100).rename('결측률(%)')
miss_df  = pd.concat([miss, miss_pct], axis=1)
miss_df  = miss_df[miss_df['결측치 수'] > 0].sort_values('결측치 수', ascending=False)
print('결측치가 있는 컬럼:')
display(miss_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
bars = ax.bar(miss_df.index, miss_df['결측치 수'], color=['#C44E52', '#4C72B0'], width=0.5)
ax.set_title('결측치 개수', fontsize=13, fontweight='bold')
ax.set_ylabel('결측치 수')
for bar, val in zip(bars, miss_df['결측치 수']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{val:,}', ha='center', fontsize=11, fontweight='bold')

ax2 = axes[1]
total      = len(df)
valid_rate = (total - miss_df['결측치 수'].sum()) / total * 100
ax2.pie(
    [valid_rate, 100 - valid_rate],
    labels=[f'유효 데이터\n{valid_rate:.1f}%', f'결측 포함\n{100-valid_rate:.1f}%'],
    colors=['#4C72B0', '#C44E52'], autopct='%1.1f%%',
    startangle=90, textprops={'fontsize': 11},
)
ax2.set_title('전체 결측률 (행 기준)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

**결측치 처리 방침**
- `loan_int_rate` (3,116개, 9.6%): 목표변수이므로 **해당 행 전체 삭제**
- `person_emp_length` (895개, 2.7%): **중앙값(median)으로 대체** (재직 중 미입력 케이스 존재)

---
## 3. 목표변수 분포 (`loan_int_rate`)

In [ ]:
rate = df['loan_int_rate'].dropna()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

ax = axes[0]
ax.hist(rate, bins=40, color='#4C72B0', edgecolor='white', linewidth=0.4)
ax.axvline(rate.mean(),   color='red',    linestyle='--', lw=1.5, label=f'평균 {rate.mean():.2f}%')
ax.axvline(rate.median(), color='orange', linestyle='--', lw=1.5, label=f'중앙값 {rate.median():.2f}%')
ax.set_title('대출 금리 분포 (히스토그램)', fontsize=12, fontweight='bold')
ax.set_xlabel('대출 금리 (%)')
ax.set_ylabel('빈도')
ax.legend()

ax2 = axes[1]
rate.plot.kde(ax=ax2, color='#4C72B0', lw=2)
ax2.set_title('대출 금리 밀도 (KDE)', fontsize=12, fontweight='bold')
ax2.set_xlabel('대출 금리 (%)')

ax3 = axes[2]
ax3.boxplot(rate, vert=True, patch_artist=True,
            boxprops=dict(facecolor='#4C72B0', alpha=0.7),
            medianprops=dict(color='red', lw=2))
ax3.set_title('대출 금리 박스플롯', fontsize=12, fontweight='bold')
ax3.set_ylabel('대출 금리 (%)')
ax3.set_xticks([])

plt.suptitle('목표변수 (loan_int_rate) 분포 분석', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'평균: {rate.mean():.4f}%   중앙값: {rate.median():.4f}%   표준편차: {rate.std():.4f}')
print(f'최솟값: {rate.min():.2f}%   최댓값: {rate.max():.2f}%   범위: {rate.max()-rate.min():.2f}%p')

---
## 4. 수치형 변수 분포

In [ ]:
num_cols = ['person_age', 'person_income', 'person_emp_length',
            'loan_amnt', 'loan_percent_income', 'cb_person_cred_hist_length']

labels_kr = {
    'person_age':               '나이 (세)',
    'person_income':            '연소득 (달러)',
    'person_emp_length':        '재직기간 (년)',
    'loan_amnt':                '대출금액 (달러)',
    'loan_percent_income':      '소득 대비 대출 비율',
    'cb_person_cred_hist_length': '신용이력 기간 (년)',
}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for ax, col in zip(axes, num_cols):
    data = df[col].dropna()
    ax.hist(data, bins=35, color='#4C72B0', edgecolor='white', linewidth=0.3)
    ax.axvline(data.mean(), color='red', linestyle='--', lw=1.3, label=f'평균 {data.mean():.1f}')
    ax.set_title(labels_kr[col], fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    if col == 'person_income':
        ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))

plt.suptitle('수치형 변수 분포', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 5. 범주형 변수 분포

In [ ]:
cat_cols = ['person_home_ownership', 'loan_intent', 'loan_grade',
            'loan_status', 'cb_person_default_on_file']

labels_kr_cat = {
    'person_home_ownership':    '주거 형태',
    'loan_intent':              '대출 목적',
    'loan_grade':               '대출 등급',
    'loan_status':              '대출 상태 (0=정상, 1=부도)',
    'cb_person_default_on_file': '과거 부도 이력 (N=없음, Y=있음)',
}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes   = axes.flatten()
colors = sns.color_palette('Set2', 8)

for ax, col in zip(axes, cat_cols):
    vc   = df[col].value_counts()
    bars = ax.bar(vc.index.astype(str), vc.values,
                  color=colors[:len(vc)], edgecolor='white')
    ax.set_title(labels_kr_cat[col], fontsize=11, fontweight='bold')
    ax.set_ylabel('빈도')
    for bar, val in zip(bars, vc.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f'{val:,}', ha='center', fontsize=8)

axes[-1].set_visible(False)
plt.suptitle('범주형 변수 분포', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 6. 피처 vs 목표변수 관계 분석

In [ ]:
df_valid    = df.dropna(subset=['loan_int_rate'])
grade_order = ['A','B','C','D','E','F','G']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
palette   = plt.cm.RdYlGn_r(np.linspace(0.15, 0.85, 7))

# 박스플롯
data_by_grade = [df_valid[df_valid['loan_grade']==g]['loan_int_rate'].values for g in grade_order]
bp = axes[0].boxplot(data_by_grade, labels=grade_order, patch_artist=True,
                     medianprops=dict(color='red', lw=2))
for patch, color in zip(bp['boxes'], palette):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[0].set_title('대출 등급별 금리 분포 (박스플롯)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('대출 등급')
axes[0].set_ylabel('대출 금리 (%)')

# 평균 금리 막대
mean_rate = df_valid.groupby('loan_grade')['loan_int_rate'].mean().reindex(grade_order)
bars      = axes[1].bar(grade_order, mean_rate.values, color=palette, edgecolor='white', width=0.6)
axes[1].set_title('대출 등급별 평균 금리', fontsize=12, fontweight='bold')
axes[1].set_xlabel('대출 등급')
axes[1].set_ylabel('평균 금리 (%)')
for bar, val in zip(bars, mean_rate.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{val:.2f}%', ha='center', fontsize=9, fontweight='bold')

plt.suptitle('대출 등급(loan_grade) vs 대출 금리(loan_int_rate)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 수치형 피처 vs 금리 산점도 (샘플 3,000개)
scatter_cols   = ['person_age', 'person_income', 'loan_amnt', 'loan_percent_income']
scatter_labels = ['나이 (세)', '연소득 (달러)', '대출금액 (달러)', '소득 대비 비율']

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
sample    = df_valid.sample(3000, random_state=42)

for ax, col, label in zip(axes, scatter_cols, scatter_labels):
    ax.scatter(sample[col], sample['loan_int_rate'], alpha=0.25, s=8, color='#4C72B0')
    ax.set_xlabel(label)
    ax.set_ylabel('대출 금리 (%)')
    ax.set_title(f'{label}\nvs 대출 금리', fontsize=10, fontweight='bold')
    if col == 'person_income':
        ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))

plt.suptitle('수치형 피처 vs 대출 금리 산점도', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 부도 이력 / 현재 상태별 금리 비교
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

meta = [
    ('cb_person_default_on_file', ['없음(N)', '있음(Y)'], '과거 부도 이력별 금리'),
    ('loan_status',               ['정상(0)', '부도(1)'], '현재 대출 상태별 금리'),
]

for ax, (col, labels, title) in zip(axes, meta):
    groups = [df_valid[df_valid[col]==v]['loan_int_rate'].dropna()
              for v in df_valid[col].unique()]
    bp = ax.boxplot(groups, labels=labels, patch_artist=True,
                    medianprops=dict(color='red', lw=2))
    for patch, c in zip(bp['boxes'], ['#55A868','#C44E52']):
        patch.set_facecolor(c)
        patch.set_alpha(0.7)
    ax.set_ylabel('대출 금리 (%)')
    ax.set_title(title, fontsize=12, fontweight='bold')

plt.suptitle('범주형 피처 vs 대출 금리 분포', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 7. 상관관계 분석 (다중공선성)

In [ ]:
from sklearn.preprocessing import LabelEncoder

df_enc = df_valid.copy()
for col in ['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']:
    df_enc[col] = LabelEncoder().fit_transform(df_enc[col].astype(str))

corr = df_enc.corr(numeric_only=True)

col_kr = {
    'person_age':               '나이',
    'person_income':            '연소득',
    'person_home_ownership':    '주거형태',
    'person_emp_length':        '재직기간',
    'loan_intent':              '대출목적',
    'loan_grade':               '대출등급',
    'loan_amnt':                '대출금액',
    'loan_int_rate':            '대출금리',
    'loan_status':              '대출상태',
    'loan_percent_income':      '소득대비비율',
    'cb_person_default_on_file': '부도이력',
    'cb_person_cred_hist_length': '신용이력',
}
corr.rename(index=col_kr, columns=col_kr, inplace=True)

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='RdYlBu_r',
    linewidths=0.3, linecolor='white', ax=ax,
    annot_kws={'size': 8}, vmin=-1, vmax=1,
    cbar_kws={'shrink': 0.8},
)
ax.set_title('피처 간 상관관계 히트맵', fontsize=13, fontweight='bold', pad=14)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# 목표변수(대출금리)와의 상관계수 순위
target_corr = corr['대출금리'].drop('대출금리').sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#C44E52' if v > 0 else '#4C72B0' for v in target_corr.values]
bars   = ax.barh(target_corr.index, target_corr.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('상관계수')
ax.set_title('각 피처와 대출 금리(loan_int_rate)의 상관계수', fontsize=12, fontweight='bold')
for bar, val in zip(bars, target_corr.values):
    offset = 0.005 if val >= 0 else -0.005
    ha     = 'left' if val >= 0 else 'right'
    ax.text(val + offset, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', ha=ha, fontsize=9)
plt.tight_layout()
plt.show()

print('\n[목표변수와 상관계수 Top 5]')
print(target_corr.head())

---
## 8. 주요 인사이트 정리

In [ ]:
print('=' * 55)
print('EDA 주요 인사이트 요약')
print('=' * 55)

print(f"""
데이터 규모
  · 원본: {len(df):,}행 x {df.shape[1]}컬럼
  · 결측치: loan_int_rate {df['loan_int_rate'].isna().sum():,}개,
            person_emp_length {df['person_emp_length'].isna().sum():,}개

목표변수 (loan_int_rate)
  · 평균: {df['loan_int_rate'].mean():.2f}%  중앙값: {df['loan_int_rate'].median():.2f}%
  · 범위: {df['loan_int_rate'].min():.2f}% ~ {df['loan_int_rate'].max():.2f}%
  · 분포: 우측 꼬리가 약간 긴 형태

핵심 피처
  · loan_grade (대출 등급): 금리와 가장 강한 양의 상관관계
    - A등급 평균 ~7%  →  G등급 평균 ~20% 이상
  · loan_amnt (대출금액): 높을수록 금리 소폭 상승 경향
  · loan_percent_income: 소득 대비 부담이 클수록 고금리

전처리 필요 사항
  · 범주형 변수 4개 → LabelEncoding
  · person_age > 100, person_emp_length > 60 이상치 제거
  · 결측치 처리 후 유효 행: 약 29,459행 예상

모델 선정 방향
  · 등급별 금리 분산이 크고 비선형 패턴 존재
    → 선형 모델보다 트리 기반 앙상블 모델이 유리할 것으로 예상
""")